In [ ]:
import pandas as pd

import os
from dotenv import load_dotenv
load_dotenv()

In [ ]:
PATH_DATA = os.getenv("PATH_DATA")

df = pd.read_csv(f"{PATH_DATA}/earthquake_data.csv")
df

In [ ]:
# Liste des colonnes à supprimer (inutiles ou géographiques)
colonnes_a_supprimer = [
    'title', 'date_time', 'net', 'magType', 'latitude', 'longitude', 'location', 'continent', 'country'
]

# On retire les colonnes si elles existent
df_clean = df.drop(columns=[col for col in colonnes_a_supprimer if col in df.columns])
df_clean.head()

In [ ]:
# Jeu de données pour la prédiction de 'alert'
df_alert = df_clean.dropna(subset=['alert'])

# Jeu de données pour la prédiction de 'sig'
df_sig = df_clean.drop(columns=['alert']) if 'alert' in df_clean.columns else df_clean.copy()

df_alert.head(), df_sig.head()

In [ ]:
from sklearn.preprocessing import StandardScaler

# Normalisation des colonnes numériques pour df_alert
colonnes_numeriques_alert = df_alert.select_dtypes(include=['float64', 'int64']).columns
scaler_alert = StandardScaler()
df_alert_scaled = df_alert.copy()
df_alert_scaled[colonnes_numeriques_alert] = scaler_alert.fit_transform(df_alert[colonnes_numeriques_alert])

# Normalisation des colonnes numériques pour df_sig
colonnes_numeriques_sig = df_sig.select_dtypes(include=['float64', 'int64']).columns
scaler_sig = StandardScaler()
df_sig_scaled = df_sig.copy()
df_sig_scaled[colonnes_numeriques_sig] = scaler_sig.fit_transform(df_sig[colonnes_numeriques_sig])

In [ ]:
from sklearn.model_selection import train_test_split

X_alert = df_alert_scaled.drop(columns=['alert'])
y_alert = df_alert_scaled['alert']
X_train_a, X_test_a, y_train_a, y_test_a = train_test_split(X_alert, y_alert, test_size=0.2, random_state=42, stratify=y_alert)

X_sig = df_sig_scaled.drop(columns=['sig'])
y_sig = df_sig_scaled['sig']
X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(X_sig, y_sig, test_size=0.2, random_state=42)

In [ ]:
X_alert.head()

In [ ]:
X_sig.head()

In [ ]:
import joblib

model_alert = joblib.load("/home/camille/code/Camille9999/EPSI/M2/EPSI_atelier-info-doc_projet/models/model_alert_logreg.joblib")
model_sig = joblib.load("/home/camille/code/Camille9999/EPSI/M2/EPSI_atelier-info-doc_projet/models/model_sig_rf.joblib")

In [ ]:
import numpy as np
from sklearn.metrics import mean_squared_error

# y_pred_a = model_alert.predict(X_test_a)
# rmse_clean_a = np.sqrt(mean_squared_error(y_test_a, y_pred_a))
# print(rmse_clean_a)

y_pred_s = model_sig.predict(X_test_s)
rmse_clean_s = np.sqrt(mean_squared_error(y_test_s, y_pred_s))
print(rmse_clean_s)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt


def evaluate_noise_impact(model, X_base, y_true, feature_idx, noise_level, scaler, rmse_clean):
    """
    Crée une copie de X, bruite une seule feature, et retourne la MSE.
    """
    X_noisy = X_base.copy()

    std_dev = X_base[:, feature_idx].std()

    noise = np.random.normal(loc=0, scale=std_dev * noise_level / 100, size=X_noisy.shape[0])
    X_noisy[:, feature_idx] += noise
    X_noisy = scaler.transform(X_noisy)

    preds_noisy = model.predict(X_noisy)
    rmse_noisy = np.sqrt(mean_squared_error(preds_noisy, y_true))
    variation = 100 * (rmse_noisy - rmse_clean) / rmse_clean
    return variation


def plot_noise_impact(intensities, mse_values, feature_names):
    plt.figure(figsize=(10, 6))
    for idx, mse in enumerate(mse_values):
        plt.plot(intensities, mse, 'o-', label=f"Bruit sur {feature_names[idx]} (%)")
    plt.title("Impact du bruit sur les features")
    plt.xlabel("Intensité du bruit (%)")
    plt.ylabel("Variation du RMSE (%)")
    plt.legend()
    plt.grid()
    plt.show()

# Affichage de la courbe
intensities = np.array([1, 3, 5, 10, 15, 20])

for idx in range(X_sig.shape[1]):
    mse_values_s = [evaluate_noise_impact(model_sig, X_sig.values, y_sig.values, idx, n, scaler_sig, rmse_clean_s) for n in intensities]

plot_noise_impact(intensities, [mse_values_s], [X_sig.columns[idx]])